<a href="https://colab.research.google.com/github/buseerarslan/Akdeniz-Otel-RAG-2026/blob/main/Akdeniz_Otel_RAG_Asistani_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏨 Akdeniz Otel ve Resort 2026 - Akıllı RAG & Multi-Modal Asistan
**Proje Amacı:** Bu çalışmada, otel misafir rehberi (PDF) ve oda fiyat listesi (CSV) üzerinden doğal dille sorulan sorulara doğru, güncel ve halüsinasyonsuz yanıtlar veren 7 adımlı bir RAG (Retrieval-Augmented Generation) ve Tool-Calling mimarisi geliştirilmiştir.

---

## 🛠️ Mimari Adımlar ve Seçim Gerekçeleri

| Adım | İşlem | Kullanılan Teknoloji / Yöntem | Neden Bu Yöntem Seçildi? (Gerekçe) |
| :--- | :--- | :--- | :--- |
| **Adım 1** | **Hazırlık & Halüsinasyon Testi** | `Gemini 2.5 Flash` | Modelin dokümansız ortamda ne kadar uydurma (halüsinasyon) yanıt verdiğini tespit etmek ve RAG sonrası başarısını kıyaslamak için. |
| **Adım 2** | **Doküman Yükleme & Metadata** | `pypdf`, LangChain `Document` | PDF sayfaları metne dönüştürülürken kaynak, sayfa no ve bölüm etiketleri eklendi. Bu sayede modele kaynak gösterme yeteneği kazandırıldı. |
| **Adım 3** | **Chunking (Parçalama)** | `RecursiveCharacterTextSplitter` *(Size: 500, Overlap: 50)* | Otel kuralları kısa maddelerden oluştuğu için paragraf bütünlüğünü bozmadan anlamsal parçalar elde etmek için tercih edildi. |
| **Adım 4** | **Vektörleştirme & Veritabanı** | `ChromaDB`, `paraphrase-multilingual-MiniLM-L12-v2` | Türkçe anlamsal aramada yüksek performans gösteren çok dilli model seçildi. Metadata filtreleme testi ile spesifik bölüm sorgusu doğrulandı. |
| **Adım 5** | **Retrieval Stratejileri** | `Similarity`, `MMR`, `Threshold` | Farklı arama teknikleri kıyaslandı. MMR ile metin çeşitliliği sağlanırken, Similarity ile doğrudan en yakın bağlam yakalandı. |
| **Adım 6** | **LCEL RAG Zinciri & Test** | LangChain Expression Language (LCEL) | Esnek, modüler ve okunabilir bir RAG zinciri kuruldu. 10 farklı soru tipi üzerinde dokümansız vs. RAG karşılaştırma tablosu üretildi. |
| **Adım 7** | **Araç Kullanımı (Tool Calling)** | LangChain `@tool`, `Pandas` | PDF'te olmayan yapısal veriler (Oda fiyatları) için CSV okuma aracı entegre edildi. Hibrit bir soru-cevap mimarisi sağlandı. |

🟢 HÜCRE 0: Kurulumlar ve Veri Setinin Otomatik Üretilmesi

In [ ]:
# Proje Bağımlılıkları (Requirements)
!pip install -qU langchain langchain-community langchain-huggingface langchain-chroma pypdf pandas sentence-transformers google-genai

In [ ]:
# ==========================================
# ADIM 0: Kütüphane Kurulumları ve Veri Üretimi
# ==========================================
!pip install -q langchain langchain-community langchain-text-splitters langchain-chroma langchain-huggingface pypdf pandas fpdf2

import os
import pandas as pd
from fpdf import FPDF

# 1. CSV Tablosunun Oluşturulması (Adım 7 Araç Sorgusu İçin Yapısal Veri)
os.makedirs("data", exist_ok=True)
csv_yolu = "otel_fiyatlari.csv"
csv_data = {
    "Oda_Tipi": ["Standart Oda", "Lüks Oda", "Aile Süiti", "Kral Dairesi", "Bungalow Ev", "Junior Süit"],
    "Kapasite": ["2 Yetişkin", "2 Yetişkin + 1 Çocuk", "4 Yetişkin", "4 Yetişkin + 2 Çocuk", "3 Yetişkin", "2 Yetişkin + 1 Bebek"],
    "Sezon_Fiyati_TL": [2500, 3800, 5200, 12000, 4100, 4500],
    "Manzara": ["Kara Manzaralı", "Deniz Manzaralı", "Deniz Manzaralı", "Panoramik Deniz", "Bahçe Manzaralı", "Deniz Manzaralı"],
    "Dahil_Hizmetler": ["Kahvaltı", "Her Şey Dahil", "Her Şey Dahil", "Ultra Her Şey Dahil", "Yarım Pansiyon", "Her Şey Dahil"]
}
pd.DataFrame(csv_data).to_csv(csv_yolu, index=False, encoding="utf-8-sig")

# 2. Tam 3 Sayfa Metin Tabanlı PDF Dokümanının Oluşturulması (Türkçe Karakter Uyumlu)
pdf_yolu = "turizm_rehberi.pdf"
pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15)

def sayfa_ekle(metin_listesi):
    pdf.add_page()
    pdf.set_font("Helvetica", size=10)
    for p in metin_listesi:
        pdf.multi_cell(0, 6, p)
        pdf.ln(3)

# Türkçe karakter içermeyen tam uyumlu metinler (FPDF çökmesini önler)
s1 = [
    "AKDENIZ OTEL VE RESORT MISAFIR EL KITABI 2026",
    "BOLUM 1: GENEL OTEL KURALLARI VE KONAKLAMA HIZMETLERI",
    "1.1. Giris ve Cikis Saatleri: Otelimize giris (check-in) saati en erken 14:00, cikis (check-out) saati ise en gec 12:00dir. Saat 12:00den sonra yapilan gec cikislarda gunluk oda ucretinin %50si ilave ek ucret olarak tahsil edilir. Saat 16:00 sonrasindaki cikislarda ise tam gun oda ucreti yansitilir.",
    "1.2. Yemek Saatleri ve Konsept: Ana restoranda acik bufe kahvalti 07:00 - 10:30 saatleri arasindadir. Gec kahvalti 10:30 - 11:30 arasinda servis edilir. Ogle yemegi 12:30 - 14:30, aksam yemegi ise 19:00 - 21:30 saatleri arasinda sunulmaktadir. Gece corbasi ve hafif bufe 23:30 - 01:00 saatleri arasinda Ana Restoranda servis edilir. Her sey dahil konseptindeki yerli alkollu ve alkolsuz icecekler 10:00 - 24:00 saatleri arasinda ucretsizdir. Yabanci ve ithal icecekler ucrete tabidir.",
    "1.3. Evcil Hayvan Politikasi: Tesisimize sadece 8 kg'a kadar olan kedi ve kopek turu evcil hayvanlar kabul edilmektedir. Evcil hayvan basina gunluk 350 TL ek temizlik ucreti uygulanir. Evcil hayvanlarin otel genel alanlarinda tasmali gezdirilmesi zorunludur. Havuz, plaj ve restoran alanlarina evcil hayvan cikarilmasi kesinlikle yasaktir."
]

s2 = [
    "AKDENIZ OTEL VE RESORT MISAFIR EL KITABI 2026",
    "BOLUM 2: AKTIVITELER, SPA VE HAVUZ KULLANIMI",
    "2.1. Havuz ve Plaj Kullanim Kurallari: Acik havuzlarimiz 08:00 - 19:00 saatleri arasinda hizmet vermektedir. Saatin ardindan hijyen ve ilaclama calismalari yapildigi icin havuza girmek can guvenligi ve saglik acisindan yasaktir. Havuz basinda ve plajda sezlong rezervasyonu icin havlu birakilmasi yasaktir; saat 09:00a kadar bos duran havlular gorevlilerce toplanir. Plaj havlulari Spa resepsiyonundan havlu karti karsiliginda temin edilir. Kayip havlu karti durumunda 200 TL kart tazminat ucreti alinir.",
    "2.2. Spa ve Wellness Hizmetleri: Sauna ve Turk hamami kullanimi 09:00 - 20:00 saatleri arasinda tum konuklarimiza ucretsizdir. Kese, kopuk, aromaterapi ve masaj hizmetleri ekstra ucrete tabidir ve en az 2 saat oncesinden Spa resepsiyonundan randevu alinmalidir. Iptal edilen randevularda 1 saat kala yapilan bildirimlerde %50 ceza kesintisi uygulanir.",
    "2.3. Cocuk Kulubu (Mini Club): 4-12 yas arasi cocuk misafirlerimiz icin Mini Club 10:00 - 12:30 ve 14:30 - 17:30 saatleri arasinda sertifikali animatorler esliginde aciktir. 4 yas alti cocuklarin ebeveynsiz katilimi kabul edilmemektedir."
]

s3 = [
    "AKDENIZ OTEL VE RESORT MISAFIR EL KITABI 2026",
    "BOLUM 3: IPTAL IADE KOSULLARI VE SPOR FAALIYETLERI",
    "3.1. Rezervasyon Iptal ve Degisiklik: Giris tarihinden 14 gun oncesine kadar yapilan iptallerde odenen tutarin %100u iade edilir. Giris tarihine 7-13 gun kala yapilan iptallerde ilk gece konaklama ucreti kesilerek kalan tutar iade edilir. Giris tarihine 7 gunden az kalan iptallerde veya tesise gelinmemesi (No-Show) durumunda odeme iadesi yapilmaz.",
    "3.2. Fitness Salonu Kullanim Kurallari: Fitness salonumuz 07:00 - 21:00 saatleri arasinda ucretsiz olarak tum misafirlerimizin kullanimina aciktir. Salona spor ayakkabisi ve havlu olmadan girilmesi kesinlikle yasaktir. 16 yasindan kucuk cocuklarin salona tek baslarina girmelerine emniyet gerekcesiyle izin verilmez.",
    "3.3. Acil Durum ve Revir: Tesisimizde 24 saat saglik personeli ve hemsire bulunmaktadir. Doktor hizmeti ucretli ve cagri uzerine hizmet vermektedir. Acil durumlarda odadaki telefonlardan 0 tuslanarak resepsiyona veya 9 tuslanarak dogrudan revire ulasilabilir."
]

sayfa_ekle(s1)
sayfa_ekle(s2)
sayfa_ekle(s3)

pdf.output(pdf_yolu)

print("✅ 'turizm_rehberi.pdf' (3 Sayfa) ve 'otel_fiyatlari.csv' sorunsuz bir sekilde olusturuldu.")

✅ 'turizm_rehberi.pdf' (3 Sayfa) ve 'otel_fiyatlari.csv' sorunsuz bir sekilde olusturuldu.


🟡 HÜCRE 1: Adım 1 - Hazırlık ve Halüsinasyon Testi

In [ ]:
# ==========================================
# ADIM 1: Hazırlık ve Halüsinasyon Testi
# ==========================================
from google.colab import ai

MODEL_NAME = "google/gemini-2.5-flash"

def sor(kullanici, sistem=None):
    prompt = kullanici
    if sistem:
        prompt = f"SİSTEM TALİMATI:\n{sistem}\n\nKULLANICI MESAJI:\n{kullanici}"
    return ai.generate_text(prompt, model_name=MODEL_NAME)

ROL = "Sen Akdeniz Otel ve Resort doküman asistanısın. Kısa ve net cevapla."

# Ödev Şartlarına Uygun 10 Test Sorusu
# (En az yarısı paraphrased, 2 tanesi dokümanda hiç yok)
SORULAR = [
    "Otelden çıkış saati kaçtır ve geç çıkışta ne kadar ücret alınır?",            # Doğrudan (Sayfa 1)
    "Tesisinize köpek getirebilir miyiz, şartları nelerdir?",                      # Paraphrased (Sayfa 1 - Evcil Hayvan)
    "Akşam yemeği hangi saatler arasında servis edilmektedir?",                    # Doğrudan (Sayfa 1)
    "Akşam havuz kaçta kapanıyor, gece yüzmek serbest mi?",                        # Paraphrased (Sayfa 2 - Havuz Kuralı)
    "Sauna ve Türk hamamı kullanımı ücretli midir?",                               # Doğrudan (Sayfa 2)
    "Tatilimi 10 gün kala iptal edersem ödediğim paranın ne kadarını geri alabilirim?", # Paraphrased (Sayfa 3 - İptal İade)
    "Odadaki çelik kasaların kullanımı ücretli midir?",                            # Doğrudan (Sayfa 3)
    "16 yaşından küçük çocuğum spor salonuna tek başına girebilir mi?",             # Paraphrased (Sayfa 3 - Fitness)
    "Otelde tenis kortu kiralama ücreti saatlik kaç TL'dir?",                      # Dokümanda YOK
    "Otelden havalimanına transfer servis araçları saat kaçta kalkıyor?"           # Dokümanda YOK
]

# Dokümansız Halüsinasyon Testi (İlk 3 Soru)
print("--- DOKÜMANSIZ MODEL CEVAPLARI (HALÜSİNASYON TESTİ) ---")
for soru_metni in SORULAR[:3]:
    cevap = sor(soru_metni, sistem=ROL)
    print(f"Soru: {soru_metni}\nCevap: {cevap}\n{'-'*50}")

--- DOKÜMANSIZ MODEL CEVAPLARI (HALÜSİNASYON TESTİ) ---
Soru: Otelden çıkış saati kaçtır ve geç çıkışta ne kadar ücret alınır?
Cevap: Standart çıkış saati öğlen 12:00'dir.

Geç çıkış talepleriniz için resepsiyonla iletişime geçmeniz gerekmektedir. Doluluk durumuna ve çıkış saatine göre ek ücretlendirme yapılabilir.
--------------------------------------------------
Soru: Tesisinize köpek getirebilir miyiz, şartları nelerdir?
Cevap: Evet, küçük ırk köpekler ek ücret karşılığında ve belirlenen alanlarda kabul edilmektedir. Ortak alanlarda tasmalı olmaları zorunludur.
--------------------------------------------------
Soru: Akşam yemeği hangi saatler arasında servis edilmektedir?
Cevap: Akşam yemeği 19:00 - 21:00 saatleri arasında servis edilmektedir.
--------------------------------------------------


🟢 HÜCRE 2: Adım 2 - Doküman Yükleme ve Metadata

In [ ]:
# ==========================================
# ADIM 2: Doküman Yükleme ve Metadata
# ==========================================
import os
from pypdf import PdfReader
from langchain_core.documents import Document

def pdf_yukle(yol, **ek_bilgi):
    """
    PDF'in her sayfasını pypdf ile okur ve metni zengin metadata ile
    LangChain Document nesnesine dönüştürür.
    """
    reader = PdfReader(yol)
    sayfalar = []

    for no, sayfa in enumerate(reader.pages, start=1):
        metin = sayfa.extract_text() or ""

        # Filtre alanının en az iki farklı değer alması için:
        bolum_adi = "genel_hizmetler" if no <= 2 else "prosedur_ve_aktivite"

        metadata = {
            "kaynak": os.path.basename(yol),
            "sayfa": no,
            "bolum": bolum_adi,
            **ek_bilgi
        }
        sayfalar.append(Document(page_content=metin, metadata=metadata))

    return sayfalar

# PDF dokümanını yükleyelim
sayfalar = pdf_yukle("turizm_rehberi.pdf", tur="otel_rehberi", yil=2026)

# Kontrol çıktıları
print(f"✅ Toplam Yüklenen Sayfa Sayısı: {len(sayfalar)}")
print(f"📄 1. Sayfa Metadata: {sayfalar[0].metadata}")
print(f"📄 3. Sayfa Metadata (Farklı Bölüm): {sayfalar[2].metadata}")

✅ Toplam Yüklenen Sayfa Sayısı: 3
📄 1. Sayfa Metadata: {'kaynak': 'turizm_rehberi.pdf', 'sayfa': 1, 'bolum': 'genel_hizmetler', 'tur': 'otel_rehberi', 'yil': 2026}
📄 3. Sayfa Metadata (Farklı Bölüm): {'kaynak': 'turizm_rehberi.pdf', 'sayfa': 3, 'bolum': 'prosedur_ve_aktivite', 'tur': 'otel_rehberi', 'yil': 2026}


🟢 HÜCRE 3: Adım 3 - Chunking Stratejisi

In [ ]:
# ==========================================
# ADIM 3: Chunking Stratejisi ve Gerekçesi
# ==========================================
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Chunking parametrelerinin tanımlanması
chunk_size = 500
chunk_overlap = 50

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Adım 2'de yüklediğimiz sayfaları parçalayalım
parcalar = splitter.split_documents(sayfalar)

# Kontrol Çıktıları
print(f"✅ Toplam Oluşan Parça (Chunk) Sayısı: {len(parcalar)}")
print("\n--- ÖRNEK CHUNK (1. PARÇA) ---")
print(f"İçerik:\n{parcalar[0].page_content}")
print(f"\nMetadata: {parcalar[0].metadata}")

✅ Toplam Oluşan Parça (Chunk) Sayısı: 9

--- ÖRNEK CHUNK (1. PARÇA) ---
İçerik:
AKDENIZ OTEL VE RESORT MISAFIR EL KITABI 2026
BOLUM 1: GENEL OTEL KURALLARI VE KONAKLAMA HIZMETLERI
1.1. Giris ve Cikis Saatleri: Otelimize giris (check-in) saati en erken 14:00, cikis (check-out) saati ise en gec 12:00dir.
Saat 12:00den sonra yapilan gec cikislarda gunluk oda ucretinin %50si ilave ek ucret olarak tahsil edilir. Saat 16:00
sonrasindaki cikislarda ise tam gun oda ucreti yansitilir.

Metadata: {'kaynak': 'turizm_rehberi.pdf', 'sayfa': 1, 'bolum': 'genel_hizmetler', 'tur': 'otel_rehberi', 'yil': 2026}


🟢 HÜCRE 4: Adım 4 - Vektör Veritabanı (Chroma) ve Metadata Filtreleme

In [ ]:
# ==========================================
# ADIM 4: Vektör Veritabanı ve Metadata Filtreleme Testi
# ==========================================
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 1. Embedding Modelinin Yüklenmesi
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Chroma Vektör Veritabanının Oluşturulması ve Parçaların Vektörleştirilmesi
vectorstore = Chroma.from_documents(
    documents=parcalar,
    embedding=embeddings,
    collection_name="otel_rehberi_collection"
)

print("✅ Vektör veritabanı başarıyla oluşturuldu ve tüm parçalar yüklendi.\n")

# 3. METADATA FİLTRELEME TESTİ (Adım 4 Zorunlu Şartı)
sorgu = "İptal ve iade koşulları nelerdir?"

print(f"🔍 Test Sorgusu: '{sorgu}'")
print("--- 1. Filtresiz Arama Sonucu ---")
filtresiz_sonuc = vectorstore.similarity_search(sorgu, k=1)
print(f"Bulunan Bölüm: {filtresiz_sonuc[0].metadata['bolum']} | Sayfa: {filtresiz_sonuc[0].metadata['sayfa']}")

print("\n--- 2. Metadata Filtreli Arama (Sadece 'prosedur_ve_aktivite' Bölümü) ---")
filtreli_sonuc = vectorstore.similarity_search(
    sorgu,
    k=1,
    filter={"bolum": "prosedur_ve_aktivite"}
)
print(f"Bulunan Bölüm: {filtreli_sonuc[0].metadata['bolum']} | Sayfa: {filtreli_sonuc[0].metadata['sayfa']}")
print(f"İçerik Özeti: {filtreli_sonuc[0].page_content[:150]}...")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Vektör veritabanı başarıyla oluşturuldu ve tüm parçalar yüklendi.

🔍 Test Sorgusu: 'İptal ve iade koşulları nelerdir?'
--- 1. Filtresiz Arama Sonucu ---
Bulunan Bölüm: prosedur_ve_aktivite | Sayfa: 3

--- 2. Metadata Filtreli Arama (Sadece 'prosedur_ve_aktivite' Bölümü) ---
Bulunan Bölüm: prosedur_ve_aktivite | Sayfa: 3
İçerik Özeti: AKDENIZ OTEL VE RESORT MISAFIR EL KITABI 2026
BOLUM 3: IPTAL IADE KOSULLARI VE SPOR FAALIYETLERI
3.1. Rezervasyon Iptal ve Degisiklik: Giris tarihinde...


🟢 HÜCRE 5: Adım 5 - Türkçe Uyumlu Retrieval Stratejileri

In [ ]:
# ==========================================
# ADIM 5: Retrieval Stratejilerinin Karşılaştırılması (Türkçe Model Uyumlu)
# ==========================================
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 1. Türkçe Anlamsal Aramada Yüksek Başarılı Multilingual Model
tr_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# Vektör veritabanını Türkçe model ile güncelliyoruz
vectorstore_tr = Chroma.from_documents(
    documents=parcalar,
    embedding=tr_embeddings,
    collection_name="otel_rehberi_tr_collection"
)

ornek_soru = "Otel odasında bulunan revire nasıl ulaşılır ve doktor hizmeti ücretli midir?"

print(f"❓ TEST SORUSU: '{ornek_soru}'\n")

# 1. Standart Similarity Search Retriever
retriever_sim = vectorstore_tr.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)
docs_sim = retriever_sim.invoke(ornek_soru)
print("--- 1. Similarity Search ---")
for i, doc in enumerate(docs_sim, start=1):
    print(f"Parça {i} (Sayfa {doc.metadata['sayfa']}): {doc.page_content[:140]}...")

# 2. MMR (Maximal Marginal Relevance) Retriever
retriever_mmr = vectorstore_tr.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 2, "fetch_k": 5, "lambda_mult": 0.5}
)
docs_mmr = retriever_mmr.invoke(ornek_soru)
print("\n--- 2. MMR Search (Çeşitlilik Odaklı) ---")
for i, doc in enumerate(docs_mmr, start=1):
    print(f"Parça {i} (Sayfa {doc.metadata['sayfa']}): {doc.page_content[:140]}...")

# 3. Similarity Score Threshold Retriever
retriever_thresh = vectorstore_tr.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.3, "k": 2}
)
docs_thresh = retriever_thresh.invoke(ornek_soru)
print("\n--- 3. Threshold Search (Eşik Değer Odaklı) ---")
for i, doc in enumerate(docs_thresh, start=1):
    print(f"Parça {i} (Sayfa {doc.metadata['sayfa']}): {doc.page_content[:140]}...")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/langchain_core/vectorstores/base.py:1048: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='21e28e57-edc2-435a-a3a1-1b3834c7252d', metadata={'tur': 'otel_rehberi', 'bolum': 'prosedur_ve_aktivite', 'kaynak': 'turizm_rehberi.pdf', 'sayfa': 3, 'yil': 2026}, page_content='uzerine hizmet vermektedir. Acil durumlarda odadaki telefonlardan 0 tuslanarak resepsiyona veya 9 tuslanarak\ndogrudan revire ulasilabilir.'), -8.413148627478147), (Document(id='61278b32-70db-4a1e-9407-21d19eea0147', metadata={'sayfa': 2, 'bolum': 'genel_hizmetler', 'kaynak': 'turizm_rehberi.pdf', 'yil': 2026, 'tur': 'otel_rehberi'}, page_content='gorevlilerce toplanir. Plaj havlulari Spa resepsiyonundan havlu karti karsiliginda temin edilir. Kayip havlu karti\ndurumunda 200 TL kart tazminat ucreti alinir.\n2.2. Spa ve Wellness Hizmetleri: Sauna ve Turk hamami kullanimi 09:00 - 20:00 saatleri arasinda tum konuklarimiza\nucretsizdir. Kese, kopuk, aromaterapi v

❓ TEST SORUSU: 'Otel odasında bulunan revire nasıl ulaşılır ve doktor hizmeti ücretli midir?'

--- 1. Similarity Search ---
Parça 1 (Sayfa 3): uzerine hizmet vermektedir. Acil durumlarda odadaki telefonlardan 0 tuslanarak resepsiyona veya 9 tuslanarak
dogrudan revire ulasilabilir....
Parça 2 (Sayfa 2): gorevlilerce toplanir. Plaj havlulari Spa resepsiyonundan havlu karti karsiliginda temin edilir. Kayip havlu karti
durumunda 200 TL kart taz...

--- 2. MMR Search (Çeşitlilik Odaklı) ---
Parça 1 (Sayfa 3): uzerine hizmet vermektedir. Acil durumlarda odadaki telefonlardan 0 tuslanarak resepsiyona veya 9 tuslanarak
dogrudan revire ulasilabilir....
Parça 2 (Sayfa 3): 3.2. Fitness Salonu Kullanim Kurallari: Fitness salonumuz 07:00 - 21:00 saatleri arasinda ucretsiz olarak tum
misafirlerimizin kullanimina a...

--- 3. Threshold Search (Eşik Değer Odaklı) ---


🟢 HÜCRE 6: Adım 6 - LCEL RAG Zinciri ve 10 Soruluk Test Tablosu

In [ ]:
# ==========================================
# ADIM 6: LCEL RAG Zinciri ve Karşılaştırmalı Test Tablosu
# ==========================================
import pandas as pd
from langchain_core.prompts import ChatPromptTemplate

# 1. RAG Prompt Şablonu
prompt_template = """Sen Akdeniz Otel ve Resort doküman asistanısın.
Aşağıda verilen BAĞLAM (doküman parçaları) bilgisini kullanarak kullanıcının sorusunu yanıtla.
Eğer verilen bağlamda sorunun cevabı YOKSA, kesinlikle bilgi uydurma ve tam olarak "Verilen dokümanda bu konuyla ilgili bir bilgi bulunmamaktadır." de.

BAĞLAM:
{context}

SORU: {question}

CEVAP:"""

prompt = ChatPromptTemplate.from_template(prompt_template)

# Context Birleştirme Fonksiyonu
def format_docs(docs):
    formatted = []
    for doc in docs:
        sayfa = doc.metadata.get("sayfa", "Bilinmiyor")
        formatted.append(f"[Kaynak: Sayfa {sayfa}]\n{doc.page_content}")
    return "\n\n".join(formatted)

# Türkçe Vektör Tabanından Retriever Tanımlama
retriever_tr = vectorstore_tr.as_retriever(search_kwargs={"k": 3})

def rag_zinciri_calistir(soru_metni):
    docs = retriever_tr.invoke(soru_metni)
    context_text = format_docs(docs)
    dolu_prompt = prompt.format(context=context_text, question=soru_metni)
    cevap = sor(dolu_prompt)
    return cevap, docs

# 2. 10 SORUNUN TEST EDİLMESİ
test_sonuclari = []

print("⏳ 10 Soru RAG Zinciri ve Dokümansız Model Üzerinde Test Ediliyor...\n")

for idx, soru_metni in enumerate(SORULAR, start=1):
    # A) Dokümansız Cevap
    dokumansiz_cevap = sor(soru_metni, sistem=ROL)

    # B) RAG İle Cevap
    rag_cevabi, getirilen_doclar = rag_zinciri_calistir(soru_metni)

    # Getirilen Sayfalar
    sayfalar_str = ", ".join(list(set([str(d.metadata.get('sayfa')) for d in getirilen_doclar])))

    test_sonuclari.append({
        "SNo": idx,
        "Soru": soru_metni,
        "Dokümansız Cevap (Gemini)": dokumansiz_cevap.replace("\n", " "),
        "RAG İle Cevap (LCEL)": rag_cevabi.replace("\n", " "),
        "Getirilen Sayfalar": sayfalar_str
    })

# DataFrame ile Tablonun Basılması
df_sonuc = pd.DataFrame(test_sonuclari)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)

print("====================================================================================================")
print("📊 RAG KARŞILAŞTIRMALI TEST TABLOSU")
print("====================================================================================================\n")
display(df_sonuc)

⏳ 10 Soru RAG Zinciri ve Dokümansız Model Üzerinde Test Ediliyor...

📊 RAG KARŞILAŞTIRMALI TEST TABLOSU



,SNo,Soru,Dokümansız Cevap (Gemini),RAG İle Cevap (LCEL),Getirilen Sayfalar
0,1,Otelden çıkış saati kaçtır ve geç çıkışta ne kadar ücret alınır?,Normal çıkış saati 12:00'dir. Geç çıkış ücretleri doluluk durumuna göre değişir: * **16:00'a kadar:** Oda ücretinin %50'si. * **16:00 sonrası:** Tam oda ücreti.,Otelden çıkış saati en geç 12:00'dir. Geç çıkış ücretleri şu şekildedir: * Saat 12:00'den sonra yapılan geç çıkışlarda günlük oda ücretinin %50'si ilave ek ücret olarak tahsil edilir. * Saat 16:00 sonrasındaki çıkışlarda ise tam gün oda ücreti yansıtılır.,"3, 1, 2"
1,2,"Tesisinize köpek getirebilir miyiz, şartları nelerdir?","Evet, küçük ırk köpekleri belirli şartlarla kabul etmekteyiz: 1. **Irk ve Kilo Limiti:** Sadece belirli küçük ırk ve kilo limitindeki köpekler kabul edilir. 2. **Aşı Karnesi:** Güncel aşı karnesinin ibrazı zorunludur. 3. **Ek Ücret:** Günlük ek evcil hayvan konaklama ücreti uygulanmaktadır. 4. **Ortak Alanlar:** Otel içinde tasmalı tutulmaları ve yiyecek-içecek alanlarına getirilmemeleri gerekmektedir.","Evet, tesisimize köpek getirebilirsiniz. Şartları şunlardır: * Sadece 8 kg'a kadar olan köpekler kabul edilmektedir. * Evcil hayvan başına günlük 350 TL ek temizlik ücreti uygulanır. * Evcil hayvanların otel genel alanlarında tasmayla gezdirilmesi zorunludur. * Havuz, plaj ve restoran alanlarına evcil hayvan çıkarılması kesinlikle yasaktır.","3, 1, 2"
2,3,Akşam yemeği hangi saatler arasında servis edilmektedir?,Akşam yemeği 19:00 - 21:30 saatleri arasında servis edilmektedir.,Akşam yemeği 19:00 - 21:30 saatleri arasında sunulmaktadır.,"1, 2"
3,4,"Akşam havuz kaçta kapanıyor, gece yüzmek serbest mi?",Havuz akşam 20:00'de kapanır. Gece yüzmek güvenlik nedeniyle yasaktır.,Akşam havuz 19:00'da kapanmaktadır. Bu saatten sonra hijyen ve ilaçlama çalışmaları yapıldığı için havuza girmek can güvenliği ve sağlık açısından yasaktır.,"3, 1, 2"
4,5,Sauna ve Türk hamamı kullanımı ücretli midir?,"Evet, ücretlidir.",Sauna ve Türk hamamı kullanımı 09:00 - 20:00 saatleri arasında tüm konuklarımıza ücretsizdir.,"1, 2"
5,6,Tatilimi 10 gün kala iptal edersem ödediğim paranın ne kadarını geri alabilirim?,"Bu, rezervasyonunuzun iptal koşullarına ve politikalarına göre değişir. Detaylar için rezervasyon anlaşmanızı kontrol etmeniz gerekmektedir.","Giriş tarihinize 7-13 gün kala yapılan iptallerde, ilk gece konaklama ücreti kesilerek kalan tutar iade edilir.","3, 1, 2"
6,7,Odadaki çelik kasaların kullanımı ücretli midir?,"Hayır, odalardaki çelik kasaların kullanımı ücretsizdir.",Verilen dokümanda bu konuyla ilgili bir bilgi bulunmamaktadır.,"3, 1"
7,8,16 yaşından küçük çocuğum spor salonuna tek başına girebilir mi?,"Hayır, 16 yaşından küçük çocuklar spor salonuna tek başına giremez.",16 yaşından küçük çocukların fitness salonuna tek başlarına girmelerine emniyet gerekçesiyle izin verilmez.,"3, 1, 2"
8,9,Otelde tenis kortu kiralama ücreti saatlik kaç TL'dir?,Bu bilgiye sahip değilim. Lütfen güncel fiyatlar için otel resepsiyonu ile iletişime geçin.,Verilen dokümanda bu konuyla ilgili bir bilgi bulunmamaktadır.,"3, 1, 2"
9,10,Otelden havalimanına transfer servis araçları saat kaçta kalkıyor?,Transfer saatleri için lütfen otel resepsiyonuyla iletişime geçiniz.,Verilen dokümanda bu konuyla ilgili bir bilgi bulunmamaktadır.,"3, 1"


🟢 HÜCRE 7: Adım 7 - CSV Araç Kullanımı (Tool Calling)

In [ ]:
# ==========================================
# ADIM 7: Structured Output ve @tool İle CSV Sorgu Aracı (DÜZELTİLDİ)
# ==========================================
import pandas as pd
from langchain_core.tools import tool

# 1. CSV Okuma Aracı Tanımlama (@tool)
@tool
def otel_fiyat_sorgula(oda_tipi: str) -> str:
    """
    Oteldeki oda tiplerine göre sezon fiyatlarını, manzarayı ve dahil olan hizmetleri sorgular.
    Girdi olarak oda tipini (örneğin: 'Lüks Oda', 'Kral Dairesi', 'Standart Oda') alır.
    """
    try:
        df = pd.read_csv("otel_fiyatlari.csv")

        # Pandas String Metodu Düzeltmesi (.str.contains)
        sonuc = df[df['Oda_Tipi'].astype(str).str.lower().str.contains(oda_tipi.lower())]

        if sonuc.empty:
            return f"'{oda_tipi}' adında bir oda tipi bulunamadı. Mevcut Oda Tipleri: {list(df['Oda_Tipi'])}"

        row = sonuc.iloc[0]
        return (f"Oda Tipi: {row['Oda_Tipi']} | Kapasite: {row['Kapasite']} | "
                f"Fiyat: {row['Sezon_Fiyati_TL']} TL | Manzara: {row['Manzara']} | "
                f"Hizmetler: {row['Dahil_Hizmetler']}")
    except Exception as e:
        return f"Hata oluştu: {str(e)}"

# 2. PDF RAG Sorgu Aracı Tanımlama (@tool)
@tool
def pdf_dokuman_sorgula(soru: str) -> str:
    """
    Akdeniz Otel ve Resort 2026 Misafir El Kitabı PDF dokümanından otel kurallarını,
    saatleri, havuz, spa ve iptal şartlarını sorgular.
    """
    cevap, docs = rag_zinciri_calistir(soru)
    return cevap

# 3. Araçların Doğrudan Test Edilmesi (.invoke)
print("--- 1. CSV Araç Testi (@tool) ---")
fiyat_sonucu = otel_fiyat_sorgula.invoke({"oda_tipi": "Kral Dairesi"})
print(f"Araç Çıktısı: {fiyat_sonucu}\n")

print("--- 2. PDF RAG Araç Testi (@tool) ---")
rag_sonucu = pdf_dokuman_sorgula.invoke({"soru": "Evcil hayvan getirme kuralı nedir?"})
print(f"Araç Çıktısı: {rag_sonucu}")

--- 1. CSV Araç Testi (@tool) ---
Araç Çıktısı: Oda Tipi: Kral Dairesi | Kapasite: 4 Yetişkin + 2 Çocuk | Fiyat: 12000 TL | Manzara: Panoramik Deniz | Hizmetler: Ultra Her Şey Dahil

--- 2. PDF RAG Araç Testi (@tool) ---
Araç Çıktısı: Akdeniz Otel ve Resort'a sadece 8 kg'a kadar olan kedi ve köpek türü evcil hayvanlar kabul edilmektedir. Evcil hayvan başına günlük 350 TL ek temizlik ücreti uygulanır. Evcil hayvanların otel genel alanlarında tasmayla gezdirilmesi zorunludur. Havuz, plaj ve restoran alanlarına evcil hayvan çıkarılması kesinlikle yasaktır.


# 🏨 Akdeniz Otel ve Resort 2026 - Akıllı RAG & Multi-Modal Asistan
**Proje Amacı:** Bu çalışmada, otel misafir rehberi (PDF) ve oda fiyat listesi (CSV) üzerinden doğal dille sorulan sorulara doğru, güncel ve halüsinasyonsuz yanıtlar veren 7 adımlı bir RAG (Retrieval-Augmented Generation) ve Tool-Calling mimarisi geliştirilmiştir.

---

## 🛠️ Mimari Adımlar ve Seçim Gerekçeleri

| Adım | İşlem | Kullanılan Teknoloji / Yöntem | Neden Bu Yöntem Seçildi? (Gerekçe) |
| :--- | :--- | :--- | :--- |
| **Adım 1** | **Hazırlık & Halüsinasyon Testi** | `Gemini 2.5 Flash` | Modelin dokümansız ortamda ne kadar uydurma (halüsinasyon) yanıt verdiğini tespit etmek ve RAG sonrası başarısını kıyaslamak için. |
| **Adım 2** | **Doküman Yükleme & Metadata** | `pypdf`, LangChain `Document` | PDF sayfaları metne dönüştürülürken kaynak, sayfa no ve bölüm etiketleri eklendi. Bu sayede modele kaynak gösterme yeteneği kazandırıldı. |
| **Adım 3** | **Chunking (Parçalama)** | `RecursiveCharacterTextSplitter` *(Size: 500, Overlap: 50)* | Otel kuralları kısa maddelerden oluştuğu için paragraf bütünlüğünü bozmadan anlamsal parçalar elde etmek için tercih edildi. |
| **Adım 4** | **Vektörleştirme & Veritabanı** | `ChromaDB`, `paraphrase-multilingual-MiniLM-L12-v2` | Türkçe anlamsal aramada yüksek performans gösteren çok dilli model seçildi. Metadata filtreleme testi ile spesifik bölüm sorgusu doğrulandı. |
| **Adım 5** | **Retrieval Stratejileri** | `Similarity`, `MMR`, `Threshold` | Farklı arama teknikleri kıyaslandı. MMR ile metin çeşitliliği sağlanırken, Similarity ile doğrudan en yakın bağlam yakalandı. |
| **Adım 6** | **LCEL RAG Zinciri & Test** | LangChain Expression Language (LCEL) | Esnek, modüler ve okunabilir bir RAG zinciri kuruldu. 10 farklı soru tipi üzerinde dokümansız vs. RAG karşılaştırma tablosu üretildi. |
| **Adım 7** | **Araç Kullanımı (Tool Calling)** | LangChain `@tool`, `Pandas` | PDF'te olmayan yapısal veriler (Oda fiyatları) için CSV okuma aracı entegre edildi. Hibrit bir soru-cevap mimarisi sağlandı. |